# ETAPA 1 - Tratamento, Tipagem e EDA - `pedidos.csv` e `avaliacoes.csv`

Preparação dos dados.

| Bloco | Tabela | O que acontece |
|---|---|---|
| 1 | `pedidos.csv` | Inspeção → Tipagem → Limpeza → Features → EDA |
| 2 | `avaliacoes.csv` | Inspeção → Tipagem → Limpeza → Features → EDA |
| 3 | Join final | Merge → EDA cruzada → Salvamento |

**Output:** `dataset_tratado.csv`


---
## 0. Imports e Configuração

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Estilo global/padrão dos gráficos
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'figure.dpi'        : 110,
})

SCORE_COLORS = {1:'#EF233C', 2:'#F4A261', 3:'#FFD166', 4:'#06D6A0', 5:'#4361EE'}
PAL_5        = [SCORE_COLORS[i] for i in range(1, 6)]

# Paths
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Projeto Integrador/Banco_de_Dados_PII3_AWS'
except ImportError:
    BASE = '.'   # local — coloque todos os arquivos na mesma pasta

PATH_PEDIDOS    = f'{BASE}/pedidos.csv'
PATH_AVALIACOES = f'{BASE}/avaliacoes.csv'
PATH_OUTPUT     = f'{BASE}/dataset_tratado.csv'

print('✅ Configuração OK')


---
## 1. `pedidos.csv`

Contém informações de ciclo de vida de cada pedido: datas de compra, aprovação, despacho, entrega real e estimada.


### 1.1 Carga e inspeção inicial

In [ ]:
pedidos = pd.read_csv(PATH_PEDIDOS)

print(f'Shape       : {pedidos.shape}')
print(f'Linhas      : {len(pedidos):,}')
print(f'Colunas     : {pedidos.shape[1]}')
print()
print('Dtypes (antes do tratamento)')
print(pedidos.dtypes.to_string())
print()
print('Nulos por coluna')
nulls = pedidos.isna().sum()
pct   = (nulls / len(pedidos) * 100).round(2)
print(pd.DataFrame({'nulos': nulls, '%': pct}).to_string())


In [ ]:
pedidos.head(5)


### 1.2 Tipagem

- Colunas de **`data`** `object`. Convertemos para `datetime64`.
- Colunas de **`ID`**. Convertemos para `string`
- Coluna de `**status de entrega**`. Convertemos para categorico

In [ ]:
pedidos = pedidos.copy()

# ── IDs: StringDtype — identificadores, não valores numéricos ───────────────
for col in ['order_id', 'customer_id']:
    pedidos[col] = pedidos[col].astype('string')

# ── Categórico: valores fixos e finitos ─────────────────────────────────────
pedidos['order_status'] = pedidos['order_status'].astype('category')

# ── Datas: todas as colunas de data convertidas para datetime ───────────────
DATE_COLS_PED = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]
for col in DATE_COLS_PED:
    pedidos[col] = pd.to_datetime(pedidos[col], errors='coerce')

print('── Dtypes após tipagem ──')
print(pedidos.dtypes.to_string())

### 1.3 EDA - Exploração da tabela antes da limpeza

Entendemos a distribuição dos dados brutos: status dos pedidos, presença de nulos e amplitude das datas.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('EDA Pedidos — visão bruta', fontsize=13, fontweight='bold')

# ── Gráfico 1: distribuição de status ──────────────────────────────────────
status_counts = pedidos['order_status'].value_counts()
cores_status  = ['#4361EE' if s == 'delivered' else '#8D99AE'
                 for s in status_counts.index]
axes[0].barh(status_counts.index, status_counts.values,
             color=cores_status, edgecolor='white')
axes[0].set_title('Distribuição de Status')
axes[0].set_xlabel('Quantidade')
for i, v in enumerate(status_counts.values):
    axes[0].text(v + 200, i, f'{v:,}', va='center', fontsize=9)

# ── Gráfico 2: nulos por coluna ────────────────────────────────────────────
nulls_pct = (pedidos.isna().sum() / len(pedidos) * 100).sort_values(ascending=False)
nulls_pct  = nulls_pct[nulls_pct > 0]
cores_null = ['#EF233C' if p > 5 else '#F4A261' for p in nulls_pct.values]
bars = axes[1].barh(nulls_pct.index, nulls_pct.values,
                    color=cores_null, edgecolor='white')
axes[1].set_title('% de Nulos por Coluna')
axes[1].set_xlabel('% nulos')
axes[1].axvline(5, color='gray', linewidth=1, linestyle='--', alpha=0.6,
                label='limite 5%')
axes[1].legend(fontsize=8)
for b, v in zip(bars, nulls_pct.values):
    axes[1].text(v + 0.1, b.get_y() + b.get_height()/2,
                 f'{v:.1f}%', va='center', fontsize=9)

# ── Gráfico 3: volume de compras por mês ───────────────────────────────────
pedidos['mes_compra'] = pedidos['order_purchase_timestamp'].dt.to_period('M')
vol_mes = pedidos.groupby('mes_compra').size()
axes[2].bar(range(len(vol_mes)), vol_mes.values, color='#4361EE', edgecolor='white')
axes[2].set_title('Volume de Compras por Mês')
axes[2].set_xlabel('Mês')
axes[2].set_ylabel('Pedidos')
step = max(1, len(vol_mes) // 8)
axes[2].set_xticks(range(0, len(vol_mes), step))
axes[2].set_xticklabels([str(vol_mes.index[i]) for i in range(0, len(vol_mes), step)],
                        rotation=30, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Período coberto: {pedidos["order_purchase_timestamp"].min().date()} → '
      f'{pedidos["order_purchase_timestamp"].max().date()}')

### 1.4 Limpeza

- Filtramos apenas pedidos entregues -> order_status == 'delivered' (avaliação válida)
- Remover pedidos sem entrega -> order_delivered_customer_date notna
Impossível calcular SLA sem essa data.
Deduplicar → drop_duplicates('order_id')
Pedidos duplicados distorceriam métricas.


In [ ]:
# Passo 1: apenas entregues
print(f'Antes da limpeza                     : {len(pedidos):,}')
pedidos = pedidos[pedidos['order_status'] == 'delivered'].copy()
print(f'Após filtro delivered                : {len(pedidos):,}')

# Passo 2: tem data de entrega real
pedidos = pedidos[pedidos['order_delivered_customer_date'].notna()].copy()
print(f'Após remover sem data de entrega     : {len(pedidos):,}')

# Passo 3: dedup
antes = len(pedidos)
pedidos = pedidos.drop_duplicates('order_id').reset_index(drop=True)
print(f'Após dedup (removidos {antes - len(pedidos):,} dups)        : {len(pedidos):,}')

print(f'\nPedidos limpos: {len(pedidos):,} linhas')


### 1.5 Feature derivada — `dias_atraso`

```
dias_atraso = data_entrega_real − data_entrega_estimada
```
- **Negativo** -> chegou **antes** do prazo (frequente)  
- **Zero** -> entregue exatamente no prazo  
- **Positivo** -> chegou **atrasado**

Esta feature é usada apenas na **análise exploratória** desta etapa para evidenciar a relação entre atraso de entrega e nota de avaliação.

In [ ]:
pedidos['dias_atraso'] = (
    pedidos['order_delivered_customer_date'] - pedidos['order_estimated_delivery_date']
).dt.days

pedidos['atrasado'] = (pedidos['dias_atraso'] > 0).astype('boolean')

print('dias_atraso — estatísticas:')
print(pedidos['dias_atraso'].describe().round(1).to_string())
print()
print(f'Pedidos atrasados  : {pedidos["atrasado"].sum():,} ({pedidos["atrasado"].mean():.1%})')
print(f'Pedidos no prazo   : {(~pedidos["atrasado"]).sum():,} ({(~pedidos["atrasado"]).mean():.1%})')


### 1.6 EDA — após limpeza

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('EDA Pedidos — após limpeza', fontsize=13, fontweight='bold')

# ── Gráfico 1: histograma de dias_atraso ───────────────────────────────────
ax = axes[0]
dentro = pedidos['dias_atraso'][pedidos['dias_atraso'].between(-50, 50)]
ax.hist(dentro, bins=40, color='#4361EE', edgecolor='white', alpha=0.85)
ax.axvline(0, color='#EF233C', linewidth=1.8, linestyle='--', label='limite prazo')
ax.set_title('Distribuição de dias_atraso')
ax.set_xlabel('Dias (negativo = adiantado)')
ax.set_ylabel('Pedidos')
ax.legend(fontsize=9)
ax.text(0.98, 0.96,
        f'Mediana: {pedidos["dias_atraso"].median():.0f}d',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

# ── Gráfico 2: % atrasados por mês ─────────────────────────────────────────
ax = axes[1]
pedidos['mes'] = pedidos['order_purchase_timestamp'].dt.to_period('M')
taxa_atraso = pedidos.groupby('mes')['atrasado'].mean() * 100
ax.bar(range(len(taxa_atraso)), taxa_atraso.values,
       color='#F4A261', edgecolor='white')
ax.axhline(taxa_atraso.mean(), color='#EF233C', linewidth=1.5,
           linestyle='--', label=f'média {taxa_atraso.mean():.1f}%')
ax.set_title('% Pedidos Atrasados por Mês')
ax.set_xlabel('Mês')
ax.set_ylabel('% atrasados')
ax.legend(fontsize=9)
step2 = max(1, len(taxa_atraso) // 7)
ax.set_xticks(range(0, len(taxa_atraso), step2))
ax.set_xticklabels([str(taxa_atraso.index[i]) for i in range(0, len(taxa_atraso), step2)],
                   rotation=30, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Resumo final da tabela pedidos
print('── Schema final pedidos ──────────────────────────')
print(pedidos.dtypes.to_string())
print()
print(f'Linhas: {len(pedidos):,} | Colunas: {pedidos.shape[1]}')
print()
print('── Amostra ──')
pedidos[['order_id','order_purchase_timestamp','order_delivered_customer_date',
     'order_estimated_delivery_date','dias_atraso','atrasado']].head(4)


---
## 2. `avaliacoes.csv`

Contém notas (1–5) e comentários de texto dos clientes, além das datas de criação e resposta da avaliação.


### 2.1 Carga e inspeção inicial

In [ ]:
avaliacoes = pd.read_csv(PATH_AVALIACOES)

print(f'Shape   : {avaliacoes.shape}')
print(f'Linhas  : {len(avaliacoes):,}')
print()
print('── Dtypes (antes do tratamento) ──')
print(avaliacoes.dtypes.to_string())
print()
print('── Nulls por coluna ──')
nulls_av = avaliacoes.isna().sum()
pct_av   = (nulls_av / len(avaliacoes) * 100).round(2)
print(pd.DataFrame({'nulos': nulls_av, '%': pct_av}).to_string())
print()
print(f'Duplicatas em review_id: {avaliacoes.duplicated("review_id").sum():,}')


In [ ]:
avaliacoes.head(5)


### 2.2 Tipagem

Datas de criação e resposta chegam como `object`. `review_score` já é `int64` (correto).


In [ ]:
avaliacoes = avaliacoes.copy()

# ── IDs: StringDtype — identificadores, não valores numéricos ───────────────
for col in ['review_id', 'order_id']:
    avaliacoes[col] = avaliacoes[col].astype('string')

# ── Categórico ordenado: review_score tem hierarquia (1 < 2 < 3 < 4 < 5) ───
avaliacoes['review_score'] = pd.Categorical(
    avaliacoes['review_score'], categories=[1,2,3,4,5], ordered=True
)

# ── Datas ───────────────────────────────────────────────────────────────────
for col in ['review_creation_date', 'review_answer_timestamp']:
    avaliacoes[col] = pd.to_datetime(avaliacoes[col], errors='coerce')

print('── Dtypes após tipagem ──')
print(avaliacoes.dtypes.to_string())

### Feature derivada: flag booleana — tem comentário de texto?

In [ ]:
avaliacoes['tem_comentario'] = avaliacoes['review_comment_message'].notna().astype('boolean')
avaliacoes

### 2.3 EDA — visão geral antes da limpeza

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('EDA Avaliações — visão bruta', fontsize=13, fontweight='bold')

# ── Gráfico 1: distribuição de notas ──────────────────────────────────────
ax = axes[0]
score_dist = avaliacoes['review_score'].value_counts().sort_index()
bars = ax.bar(score_dist.index.astype(int), score_dist.values,
              color=PAL_5, edgecolor='white', linewidth=0.8)
ax.set_title('Distribuição de Notas (bruta)')
ax.set_xlabel('Nota')
ax.set_ylabel('Avaliações')
ax.set_xticks([1,2,3,4,5])
for b in bars:
    pct = b.get_height() / len(avaliacoes) * 100
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 300,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

# ── Gráfico 2: cobertura de comentários por nota ───────────────────────────
ax = axes[1]
cobertura = (avaliacoes.groupby('review_score', observed=True)['tem_comentario']
               .mean() * 100)
bars2 = ax.bar(cobertura.index.astype(int), cobertura.values,
               color=PAL_5, edgecolor='white')
ax.set_title('% com Comentário por Nota')
ax.set_xlabel('Nota')
ax.set_ylabel('% com comentário')
ax.set_xticks([1,2,3,4,5])
ax.axhline(avaliacoes['tem_comentario'].mean()*100, color='gray',
           linewidth=1.2, linestyle='--',
           label=f'média {avaliacoes["tem_comentario"].mean()*100:.1f}%')
ax.legend(fontsize=9)
for b, v in zip(bars2, cobertura.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5,
            f'{v:.0f}%', ha='center', va='bottom', fontsize=9)

# ── Gráfico 3: volume de avaliações por mês ────────────────────────────────
ax = axes[2]
avaliacoes['mes_av'] = avaliacoes['review_creation_date'].dt.to_period('M')
vol_av = avaliacoes.groupby('mes_av').size()
ax.bar(range(len(vol_av)), vol_av.values, color='#4361EE', edgecolor='white')
ax.set_title('Volume de Avaliações por Mês')
ax.set_xlabel('Mês')
ax.set_ylabel('Avaliações')
step3 = max(1, len(vol_av) // 8)
ax.set_xticks(range(0, len(vol_av), step3))
ax.set_xticklabels([str(vol_av.index[i]) for i in range(0, len(vol_av), step3)],
                   rotation=30, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

print(f'Nota mais frequente : {score_dist.idxmax()} ({score_dist.max()/len(avaliacoes):.1%} do total)')
print(f'Clientes que comentam: {avaliacoes["tem_comentario"].mean():.1%} do total')

### 2.4 Limpeza

| Passo | Critério | Motivo |
|---|---|---|
| Deduplicar | `drop_duplicates('review_id')` | Havia 814 review_id duplicados |
| Validar score | `review_score ∈ [1, 5]` | Sanidade do target |

> `review_comment_message` **não** é filtrado aqui — mantemos avaliações sem comentário porque a EDA e o join precisam delas. O filtro para o modelo fica no notebook de modelagem.


In [ ]:
# Dedup
antes = len(avaliacoes)
avaliacoes = avaliacoes.drop_duplicates('review_id').reset_index(drop=True)
print(f'Removidas {antes - len(avaliacoes):,} linhas duplicadas por review_id')
print(f'Após dedup  : {len(avaliacoes):,}')

# Validar score
avaliacoes = avaliacoes[avaliacoes['review_score'].isin([1,2,3,4,5])].copy()
print(f'Após validar score [1-5] : {len(avaliacoes):,}')

print(f'\nAvaliações limpas: {len(avaliacoes):,} linhas')


### 2.5 Feature derivada — `horas_resposta`

Tempo entre o cliente criar a avaliação e a loja/plataforma responder.  
Medido em horas (float).


In [ ]:
avaliacoes['horas_resposta'] = (
    avaliacoes['review_answer_timestamp'] - avaliacoes['review_creation_date']
).dt.total_seconds() / 3600

print('horas_resposta — estatísticas:')
print(avaliacoes['horas_resposta'].describe().round(1).to_string())
print()

# Classifica velocidade de resposta
bins   = [0, 24, 48, 120, float('inf')]
labels = ['< 1 dia', '1–2 dias', '2–5 dias', '> 5 dias']
avaliacoes['velocidade_resposta'] = pd.cut(avaliacoes['horas_resposta'], bins=bins, labels=labels)
print('Distribuição de velocidade_resposta:')
print(avaliacoes['velocidade_resposta'].value_counts().to_string())


### 2.6 EDA — após limpeza

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('EDA Avaliações — após limpeza', fontsize=13, fontweight='bold')

# ── Gráfico 1: nota média por mês ─────────────────────────────────────────
ax = axes[0]
avaliacoes['review_score'] = pd.to_numeric(avaliacoes['review_score'], errors='coerce')
avaliacoes['mes_av'] = avaliacoes['review_creation_date'].dt.to_period('M')
nota_mes = avaliacoes.groupby('mes_av')['review_score'].mean()
ax.plot(range(len(nota_mes)), nota_mes.values,
        color='#4361EE', linewidth=2, marker='o', markersize=4)
ax.fill_between(range(len(nota_mes)), nota_mes.values, alpha=0.15, color='#4361EE')
ax.axhline(nota_mes.mean(), color='#EF233C', linewidth=1.2, linestyle='--',
           label=f'média {nota_mes.mean():.2f}')
ax.set_title('Nota Média por Mês')
ax.set_xlabel('Mês')
ax.set_ylabel('Nota média')
ax.set_ylim(1, 5)
ax.legend(fontsize=9)
step4 = max(1, len(nota_mes) // 8)
ax.set_xticks(range(0, len(nota_mes), step4))
ax.set_xticklabels([str(nota_mes.index[i]) for i in range(0, len(nota_mes), step4)],
                   rotation=30, ha='right', fontsize=8)

# ── Gráfico 2: distribuição horas_resposta (clip 400h) ────────────────────
ax = axes[1]
hr_clip = avaliacoes['horas_resposta'].clip(0, 400)
ax.hist(hr_clip, bins=50, color='#06D6A0', edgecolor='white', alpha=0.85)
ax.axvline(avaliacoes['horas_resposta'].median(), color='#EF233C', linewidth=1.8,
           linestyle='--', label=f'mediana {avaliacoes["horas_resposta"].median():.0f}h')
ax.set_title('Distribuição horas_resposta (clip 400h)')
ax.set_xlabel('Horas')
ax.set_ylabel('Avaliações')
ax.legend(fontsize=9)

# ── Gráfico 3: nota média por velocidade de resposta ─────────────────────
ax = axes[2]
ordem = ['< 1 dia', '1–2 dias', '2–5 dias', '> 5 dias']
nota_veloc = (avaliacoes.groupby('velocidade_resposta', observed=True)['review_score']
                .agg(['mean','count'])
                .reindex(ordem))
cores_veloc = ['#06D6A0','#4361EE','#F4A261','#EF233C']
bars3 = ax.bar(range(len(ordem)), nota_veloc['mean'],
               color=cores_veloc, edgecolor='white')
ax.set_title('Nota Média por Velocidade de Resposta')
ax.set_xlabel('Velocidade')
ax.set_ylabel('Nota média')
ax.set_ylim(1, 5)
ax.set_xticks(range(len(ordem)))
ax.set_xticklabels(ordem, rotation=15, ha='right', fontsize=9)
for b, (_, row) in zip(bars3, nota_veloc.iterrows()):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.04,
            f'{row["mean"]:.2f}\n(n={row["count"]:,.0f})',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Resumo final da tabela avaliações
print(' Schema final avaliações ')
print(avaliacoes.dtypes.to_string())
print()
print(f'Linhas: {len(avaliacoes):,} | Colunas: {avaliacoes.shape[1]}')
print()
print(' Amostra ')
avaliacoes[['review_id','order_id','review_score','tem_comentario',
    'horas_resposta','velocidade_resposta']].head(4)


---
## 3. Join final — `avaliações e pedidos`

**Inner join** por `order_id`: mantém apenas pedidos entregues que possuem avaliação (e vice-versa).  
Registros sem par em uma das tabelas são descartados.


### 3.1 Merge

In [ ]:
df = avaliacoes.merge(
    pedidos,
    on='order_id',
    how='inner',
    suffixes=('_av', '_ped')
)

print(f'avaliações (limpas) : {len(avaliacoes):,}')
print(f'pedidos    (limpos) : {len(pedidos):,}')
print(f'join inner          : {len(df):,}  ({len(df)/len(avaliacoes):.1%} das avaliações encontraram pedido entregue)')
print()
print('── Colunas do dataset final ──')
for col in df.columns:
    print(f'  {col:<36} {str(df[col].dtype):<15}')


### 3.2 EDA cruzada — entrega × satisfação

Investigamos a relação entre o comportamento de entrega (atraso, SLA) e a nota dada pelo cliente.


In [ ]:
# Score médio por grupo de atraso
bins_at   = [-float('inf'), -1, 0, 7, 30, float('inf')]
labs_at   = ['Adiantado', 'No prazo', '1–7d atraso', '8–30d atraso', '>30d atraso']
df['grupo_atraso'] = pd.Categorical(
    pd.cut(df['dias_atraso'], bins=bins_at, labels=labs_at),
    categories=labs_at, ordered=True
)

score_grupo = (df.groupby('grupo_atraso', observed=True)['review_score']
                 .agg(['mean','count','std'])
                 .round(2))

print('Nota média por grupo de atraso:')
print(score_grupo.to_string())
print()
print(f'Correlação (Pearson) dias_atraso × review_score: '
      f'{df[["dias_atraso","review_score"]].corr().iloc[0,1]:+.3f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('EDA Cruzada — Entrega × Satisfação', fontsize=13, fontweight='bold')

# ── Gráfico 1: nota média por grupo de atraso ─────────────────────────────
ax = axes[0]
cores_at = ['#06D6A0','#4361EE','#FFD166','#F4A261','#EF233C']
bars_at  = ax.bar(range(len(score_grupo)), score_grupo['mean'],
                  color=cores_at, edgecolor='white')
ax.set_xticks(range(len(score_grupo)))
ax.set_xticklabels(score_grupo.index, rotation=20, ha='right', fontsize=9)
ax.set_title('Nota Média por Grupo de Atraso')
ax.set_ylabel('Nota média')
ax.set_ylim(1, 5.3)
for b, (_, row) in zip(bars_at, score_grupo.iterrows()):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
            f'{row["mean"]:.2f}\n(n={row["count"]:,.0f})',
            ha='center', va='bottom', fontsize=8)

# ── Gráfico 2: boxplot score × grupo de atraso ────────────────────────────
ax = axes[1]
data_bx = [df[df['grupo_atraso'] == g]['review_score'].astype(float).values
           for g in labs_at]
bp2 = ax.boxplot(data_bx, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, cor in zip(bp2['boxes'], cores_at):
    patch.set_facecolor(cor)
    patch.set_alpha(0.75)
ax.set_xticks(range(1, len(labs_at)+1))
ax.set_xticklabels(labs_at, rotation=20, ha='right', fontsize=9)
ax.set_title('Boxplot Score por Grupo de Atraso')
ax.set_ylabel('review_score')

# ── Gráfico 3: heatmap de correlações numéricas ───────────────────────────
ax = axes[2]
num_cols = ['review_score','dias_atraso','horas_resposta']
df_num   = df[num_cols].copy()
df_num['review_score'] = df_num['review_score'].astype(float)
corr_mat = df_num.corr().round(2)
mask     = np.zeros_like(corr_mat, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(corr_mat, ax=ax, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            annot_kws={'size': 12}, square=True)
ax.set_title('Correlações (Pearson)')

plt.tight_layout()
plt.show()


In [ ]:
# Proporção de notas por grupo de atraso (stacked bar)
fig, ax = plt.subplots(figsize=(11, 4))
fig.suptitle('Distribuição de Notas por Grupo de Atraso', fontsize=13, fontweight='bold')

prop = (df.groupby(['grupo_atraso','review_score'], observed=True)
          .size()
          .unstack('review_score')
          .fillna(0))
prop_pct = prop.div(prop.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(prop_pct))
for nota, cor in zip([1,2,3,4,5], PAL_5):
    if nota in prop_pct.columns:
        vals = prop_pct[nota].values
        ax.bar(range(len(prop_pct)), vals, bottom=bottom,
               label=f'Nota {nota}', color=cor, edgecolor='white', linewidth=0.5)
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 6:
                ax.text(i, b + v/2, f'{v:.0f}%', ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottom += vals

ax.set_xticks(range(len(prop_pct)))
ax.set_xticklabels(prop_pct.index, rotation=15, ha='right')
ax.set_ylabel('% de avaliações')
ax.set_ylim(0, 100)
ax.legend(loc='lower right', ncol=5, fontsize=9)
plt.tight_layout()
plt.show()


### 3.3 Schema e qualidade final do dataset

In [ ]:
print('── Schema final do dataset consolidado ───────────────────────────────────')
info = pd.DataFrame({
    'dtype'   : df.dtypes,
    'nulos'   : df.isna().sum(),
    '% nulos' : (df.isna().sum() / len(df) * 100).round(2),
    'únicos'  : df.nunique(),
})
print(info.to_string())
print()
print(f'Linhas  : {len(df):,}')
print(f'Colunas : {df.shape[1]}')


In [ ]:
# Sumário estatístico das colunas numéricas
df_num_final = df[['review_score','dias_atraso','horas_resposta']].copy()
df_num_final['review_score'] = df_num_final['review_score'].astype(float)
print('── Estatísticas descritivas (colunas numéricas-chave) ──')
print(df_num_final.describe().round(2).to_string())


### 3.4 Salvamento

Salva o dataset final em CSV para uso nos notebooks de modelagem.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Garante que categorias sejam salvas como string
df_save = df.copy()
df_save['review_score']       = df_save['review_score'].astype(int)
df_save['grupo_atraso']       = df_save['grupo_atraso'].astype(str)
df_save['velocidade_resposta']= df_save['velocidade_resposta'].astype(str)
df_save['tem_comentario']     = df_save['tem_comentario'].astype(bool)
df_save['atrasado']           = df_save['atrasado'].astype(bool)

df_save.to_csv(PATH_OUTPUT, index=False)
size_mb = os.path.getsize(PATH_OUTPUT) / 1024**2
print(f'✅ Salvo em: {PATH_OUTPUT}')
print(f'   Tamanho : {size_mb:.1f} MB')
print(f'   Linhas  : {len(df_save):,}')
print(f'   Colunas : {df_save.shape[1]}')


---
## 4. Resumo

### O que foi feito

| Tabela | Bruto | Após limpeza | Removidos | Motivo principal |
|---|---|---|---|---|
| `pedidos.csv` | 99.441 | 96.470 | 2.971 | status ≠ delivered + sem data entrega |
| `avaliacoes.csv` | 99.224 | 98.410* | 814 | duplicatas de review_id |
| **Join final** | — | **95.601** | — | inner join por order_id |

*A tabela de avaliações mantém registros sem comentário — filtro para o modelo fica no notebook de modelagem.

### Features derivadas criadas

| Feature | Tabela | Descrição |
|---|---|---|
| `dias_atraso` | pedidos | entrega_real − entrega_estimada (negativo = adiantado) |
| `atrasado` | pedidos | bool — `dias_atraso > 0` |
| `tem_comentario` | avaliacoes | bool — `review_comment_message notna` |
| `horas_resposta` | avaliacoes | horas entre criação e resposta da avaliação |
| `velocidade_resposta` | avaliacoes | categoria: < 1 dia / 1–2 dias / 2–5 dias / > 5 dias |
| `grupo_atraso` | join | categoria: Adiantado / No prazo / 1–7d / 8–30d / >30d |

### Principais achados da EDA

- **Nota 5 domina**: 57% das avaliações — dataset desbalanceado, requer atenção na modelagem.
- **Atraso destrói satisfação**: pedidos com 1–7 dias de atraso têm nota média 2,72 vs 4,30 dos adiantados.
- **Clientes insatisfeitos comentam mais**: nota 1 tem 67% de cobertura de comentário, nota 5 tem apenas 24%.
- **Correlação atraso × nota**: Pearson = −0,27 (negativa, esperada — quanto mais atraso, menor a nota).
- **A maioria dos pedidos chega adiantada**: mediana de `dias_atraso` = −12 dias.


---
# ETAPA 2 - Predição de Notas com Regressão Linear em Texto

Continuamos a partir da base tratada salva pelo EDA.  
Carregamos `dataset_tratado.csv` para que essa parte fique independente — pode ser executada sem reexecutar todo o tratamento.


In [ ]:
# # Leitura da base tratada salva ao final do EDA
# df = pd.read_csv(PATH_OUTPUT)

# print(f'Base tratada carregada: {len(df):,} linhas | {df.shape[1]} colunas')
# print(f'Colunas: {list(df.columns)}')
# df.head(3)

## 📊 Predição de Nota de Avaliação com Regressão Linear em Texto

**Objetivo:** dado o texto livre de um comentário de cliente, estimar a nota (1 a 5) que ele irá atribuir à sua compra.

**Técnica central:** transformamos palavras em números (TF-IDF) e usamos esses números como variáveis explicativas em uma regressão linear. O modelo aprende que certas palavras ("péssimo", "atrasado") puxam a nota para baixo, enquanto outras ("rápido", "perfeito") puxam para cima.

---
## Estrutura do notebook

| # | Seção | O que acontece |
|---|-------|----------------|
| 1 | Imports & Configuração | Bibliotecas e constantes globais |
| 2 | Stopwords PT-BR | Lista de palavras sem valor semântico |
| 3 | Funções Auxiliares | Limpeza de texto e predição |
| 4 | Carga dos Dados | Leitura do CSV e inspeção inicial |
| 5 | EDA | Distribuição das notas e cobertura de comentários |
| 6 | Pré-processamento | Limpeza e normalização do texto |
| 7 | Divisão Treino/Teste | Separação estratificada dos dados |
| 8 | TF-IDF | Vetorização do texto em features numéricas |
| 9 | Regressão Ridge | Treinamento e avaliação do modelo |
| 10 | Interpretação | Palavras que mais influenciam a nota |
| 11 | Visualizações | Gráficos de EDA e performance |
| 12 | Predição ao Vivo | Testando com novos comentários |
| 13 | Resumo & Próximos Passos | Conclusões e melhorias sugeridas |

---
## 1. Imports & Configuração

Todas as bibliotecas necessárias são nativas do ambiente Colab — nenhuma instalação adicional é necessária.

- **pandas / numpy** → manipulação de dados
- **matplotlib / seaborn** → visualizações
- **sklearn** → TF-IDF, Ridge Regression, métricas e pipeline
- **re** → expressões regulares para limpar o texto

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score, confusion_matrix
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
%matplotlib inline

print("✅ Imports OK")

In [ ]:
# ── Constantes globais ──────────────────────────────────────────────────────
# Altere DATA_PATH se o arquivo estiver em outro local (ex.: Google Drive)
DATA_PATH   = df   # ajuste para o caminho correto no seu ambiente
RANDOM_SEED = 42                 # garante reprodutibilidade
TEST_SIZE   = 0.20               # 20% dos dados para teste

# Paleta de cores dos gráficos
PALETTE = {
    "primary" : "#4361EE",
    "positive": "#06D6A0",
    "negative": "#EF233C",
    "neutral" : "#8D99AE",
    "bg"      : "#F8F9FA",
}
SCORE_COLORS = {1: "#EF233C", 2: "#F4A261", 3: "#FFD166",
                4: "#06D6A0", 5: "#4361EE"}

# Estilo global dos gráficos
plt.rcParams.update({
    "figure.facecolor" : PALETTE["bg"],
    "axes.facecolor"   : PALETTE["bg"],
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.family"      : "sans-serif",
})

print("✅ Configurações OK")

---
## 2. Stopwords PT-BR

**Stopwords** são palavras de alta frequência que não carregam significado útil para o modelo (artigos, preposições, pronomes, verbos auxiliares).

Exemplos: *"de", "a", "o", "que", "para", "com"*

Removê-las reduz o ruído e melhora a qualidade das features.
Aqui usamos uma lista manual em PT-BR, evitando dependências externas (como o `nltk`).

In [ ]:
STOPWORDS_PT = set([
    "de","a","o","que","e","do","da","em","um","para","com","uma","os","no",
    "se","na","por","mais","as","dos","como","mas","ao","ele","das","à","seu",
    "sua","ou","quando","muito","nos","já","eu","também","só","pelo","pela",
    "até","isso","ela","entre","depois","sem","mesmo","aos","seus","quem","nas",
    "me","esse","eles","você","essa","num","nem","suas","meu","às","minha",
    "numa","pelos","elas","seja","qual","será","nós","tenho","lhe","deles",
    "essas","esses","pelas","este","foi","ter","tem","temos","foram","está",
    "tinha","estou","vou","vai","fui","era","ser","não","sim","aqui","ali",
    "lá","então","agora","ainda","aquele","tudo","todo","todos","toda","todas",
    "cada","outro","outra","outros","outras","nada","algo","alguém","ninguém",
    "algum","alguma","alguns","algumas","nenhum","nenhuma","há","la","le","pra",
    "pro","bem","ta","tá","tb","tbm","faz","fazer","feito","ficou","ficei",
    "fique","fica","veio","vem","vir","vão","vim","ver","vi","vejo","so","ja",
    "ne","né","nao","ai","aí","ah","oh","oi","ola",
])

print(f"✅ {len(STOPWORDS_PT)} stopwords carregadas")

### Foi necessário aplicar ajustes para melhorar o desempenho:

In [ ]:
NEGACOES = {"não", "nao", "nem", "nunca", "jamais", "sem", "nada", "nenhum", "nenhuma"}
STOPWORDS_PT = STOPWORDS_PT - NEGACOES

---
## 3. Funções Auxiliares

### `clean_text(text)`
Aplica o pipeline de limpeza NLP (processamento de linguagem natural), ao texto bruto:
1. Converte para minúsculas
2. Remove tudo que não for letra (pontuação, números, emojis)
3. Remove espaços duplicados
4. Filtra stopwords e tokens muito curtos (< 3 caracteres)

### `round_to_score(pred)`
Converte a predição contínua (ex.: 3.7) para inteiro mais próximo no intervalo [1, 5].

### `predict_comment(text, pipeline)`
Recebe um texto novo e retorna a nota estimada + as palavras que mais influenciaram a decisão.

In [ ]:
def clean_text(text: str) -> str:
    """Normaliza e limpa um comentário em português."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-záàâãéèêíïóôõöúüçñ\s]', ' ', text)  # só letras
    text = re.sub(r'\s+', ' ', text).strip()                  # espaços extras
    tokens = [w for w in text.split()
              if w not in STOPWORDS_PT and len(w) > 2]
    return ' '.join(tokens)


def round_to_score(pred: np.ndarray) -> np.ndarray:
    """Arredonda predição contínua para nota inteira em [1, 5]."""
    return np.clip(np.round(pred).astype(int), 1, 5)


def predict_comment(text: str, pipeline) -> dict:
    """Retorna nota estimada e palavras-chave para um comentário novo."""
    text_clean = clean_text(text)
    tfidf_vec  = pipeline.named_steps["tfidf"]
    model      = pipeline.named_steps["ridge"]

    vec     = tfidf_vec.transform([text_clean])
    raw     = model.predict(vec)[0]
    clipped = float(np.clip(raw, 1, 5))
    rounded = int(round_to_score(np.array([clipped]))[0])

    # Contribuição de cada palavra presente no comentário
    feat_names   = tfidf_vec.get_feature_names_out()
    coef         = model.coef_
    nonzero      = vec.nonzero()[1]
    word_weights = sorted(
        [(feat_names[i], coef[i] * vec[0, i]) for i in nonzero],
        key=lambda x: x[1], reverse=True
    )
    return {
        "score_raw"    : round(clipped, 2),
        "score_rounded": rounded,
        "palavras_pos" : [(w, round(float(v), 3)) for w, v in word_weights if v > 0][:5],
        "palavras_neg" : [(w, round(float(v), 3)) for w, v in word_weights if v < 0][:5],
    }


# Teste rápido da limpeza
exemplo = "Produto chegou antes do prazo! Muito bom :)"
print(f"Original : {exemplo}")
print(f"Limpo    : {clean_text(exemplo)}")

---
## 4. Carga dos Dados

O dataset `avaliacoes.csv` contém **99.224 avaliações** de clientes.

Colunas relevantes para este projeto:

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `review_score` | int (1–5) | ⭐ **Target (y)** — nota dada pelo cliente |
| `review_comment_message` | str | 💬 **Input (X)** — texto livre do comentário |
| `review_comment_title` | str | Título opcional (pouco preenchido) |
| `review_creation_date` | str | Data da avaliação |

> ⚠️ **Importante:** apenas ~41% das avaliações têm comentário de texto. O modelo só consegue aprender a partir dessas.

In [ ]:
print(f"Total de avaliações  : {len(df):,}")
print(f"Colunas              : {list(df.columns)}")
print(f"Com comentário       : {df['review_comment_message'].notna().sum():,}")
print(f"Sem comentário       : {df['review_comment_message'].isna().sum():,}")

periodo = pd.to_datetime(df['review_creation_date'], errors='coerce')
print(f"Período              : {periodo.min().strftime('%Y-%m-%d')} --> {periodo.max().strftime('%Y-%m-%d')}")
print()
df.head(3)

---
## 5. Análise Exploratória (EDA)

Antes de modelar, precisamos entender a distribuição dos dados:

- **Distribuição das notas:** o dataset é fortemente enviesado — a nota 5 representa mais de 57% das avaliações. Isso é um **desafio para o modelo**, pois ele terá mais exemplos de notas altas.
- **Cobertura de comentários:** clientes insatisfeitos (nota 1) comentam proporcionalmente mais do que satisfeitos (nota 5), que frequentemente avaliam sem texto.
- **Tamanho dos comentários:** comentários negativos tendem a ser mais longos — o cliente insatisfeito tem mais a dizer.

In [ ]:
score_dist = df["review_score"].value_counts().sort_index()
print("Distribuição de notas:")
print(score_dist.to_string())
print(f"\nNota mais comum: {score_dist.idxmax()} ({score_dist.max()/len(df):.1%} do total)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("EDA — Avaliações de Clientes", fontsize=15, fontweight="bold")

# ── Gráfico 1: Distribuição geral das notas ──────────────────────────────────
ax = axes[0]
bars = ax.bar(score_dist.index, score_dist.values,
              color=[SCORE_COLORS[s] for s in score_dist.index],
              edgecolor="white", linewidth=0.8)
ax.set_title("Distribuição de Notas (Total)")
ax.set_xlabel("Nota")
ax.set_ylabel("Quantidade")
ax.set_xticks([1, 2, 3, 4, 5])
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 300,
            f"{int(b.get_height()):,}", ha="center", va="bottom", fontsize=9)

# ── Gráfico 2: Com vs. sem comentário por nota ───────────────────────────────
has_comment = df.groupby("review_score")["review_comment_message"].apply(
    lambda x: pd.Series({"Com comentário": x.notna().sum(),
                          "Sem comentário": x.isna().sum()})
).unstack()
has_comment.plot(kind="bar", ax=axes[1],
                 color=[PALETTE["primary"], PALETTE["neutral"]],
                 edgecolor="white", linewidth=0.8, rot=0)
axes[1].set_title("Com vs. Sem Comentário por Nota")
axes[1].set_xlabel("Nota")
axes[1].set_ylabel("Quantidade")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Pré-processamento de Texto

Aplicamos a função `clean_text` a todos os comentários e filtramos registros vazios após a limpeza.

**Passos aplicados a cada comentário:**
```
"Produto chegou antes do prazo! Muito bom :)"
        ↓ lower()
"produto chegou antes do prazo! muito bom :)"
        ↓ remove não-letras
"produto chegou antes do prazo  muito bom  "
        ↓ remove stopwords e tokens < 3 chars
"produto chegou antes prazo bom"
```

**Observação interessante:** clientes que deixam notas baixas (1–2) escrevem comentários mais longos em média — eles têm mais a reclamar.

In [ ]:
# Filtra apenas linhas com comentário e aplica limpeza
df_model = df[df["review_comment_message"].notna()].copy()
df_model["text_clean"] = df_model["review_comment_message"].apply(clean_text)

# Remove comentários que ficaram vazios após a limpeza
df_model = df_model[df_model["text_clean"].str.len() > 3].reset_index(drop=True)

print(f"Registros após filtro : {len(df_model):,}")
print()
print("Exemplo de limpeza:")
idx = df_model[df_model["review_score"] == 1].index[0]
print(f"  Original : {df_model.loc[idx, 'review_comment_message']}")
print(f"  Limpo    : {df_model.loc[idx, 'text_clean']}")

In [ ]:
# Contagem de palavras por avaliação (após limpeza)
df_model["word_count"] = df_model["text_clean"].str.split().str.len()
avg_words = df_model.groupby("review_score")["word_count"].mean().round(1)

print("Média de palavras por nota (após limpeza):")
print(avg_words.to_string())

# Atualiza o 3º gráfico da EDA com os dados reais
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(avg_words.index, avg_words.values,
       color=[SCORE_COLORS[s] for s in avg_words.index],
       edgecolor="white")
ax.set_title("Média de Palavras por Nota (após limpeza)")
ax.set_xlabel("Nota")
ax.set_ylabel("Nº de palavras")
ax.set_xticks([1, 2, 3, 4, 5])
for i, v in zip(avg_words.index, avg_words.values):
    ax.text(i, v + 0.1, f"{v:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 7. Divisão Treino / Teste

Dividimos o dataset em dois conjuntos:
- **Treino (80%):** usado para o modelo aprender os padrões.
- **Teste (20%):** reservado para avaliar o modelo em dados **nunca vistos**.

O parâmetro `stratify=y` garante que a proporção de cada nota seja mantida nos dois conjuntos — essencial quando há desequilíbrio de classes (como aqui, onde nota 5 é muito mais frequente).

```
stratify=y
    Nota 5: 57% treino  →  57% teste   ✅
    Nota 1: 22% treino  →  22% teste   ✅
```

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df_model["text_clean"],
    df_model["review_score"],
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df_model["review_score"],  # mantém proporção de classes
)

print(f"Treino : {len(X_train):,} amostras")
print(f"Teste  : {len(X_test):,} amostras")
print()
print("Proporção de notas no treino:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(1).to_string())

---
## 8. Vetorização com TF-IDF

**Modelos de Machine Learning não entendem texto — só números.**
O TF-IDF converte cada comentário em um vetor numérico.

### O que é TF-IDF?

Para cada palavra em cada documento, calcula dois fatores:

- **TF (Term Frequency):** quão frequente é a palavra *neste* documento.
- **IDF (Inverse Document Frequency):** quão rara ela é *em todos* os documentos — palavras comuns demais recebem peso menor.

```
TF-IDF(palavra, doc) = TF × IDF
```

Resultado: uma matriz esparsa onde cada **linha = comentário** e cada **coluna = palavra (feature)**.

### Parâmetros escolhidos

| Parâmetro | Valor | Por quê |
|-----------|-------|---------|
| `max_features` | 8.000 | Limita vocabulário ao top-8k palavras mais informativas |
| `ngram_range=(1,2)` | uni + bigramas | Captura "chegou antes", "produto quebrado" como features únicas |
| `min_df=5` | min 5 docs | Ignora palavras que aparecem em menos de 5 comentários (ruído) |
| `sublinear_tf=True` | log(TF) | Evita que palavras repetidas muitas vezes dominem o vetor |

> 💡 **Bigrama** = sequência de 2 palavras. Ex: "entrega rápida", "não recebi". Capturam contexto que unigramas isolados perderiam.

---
## 9. Treinamento — Ridge Regression

### Por que Ridge e não Regressão Linear simples?

Com 8.000 features (palavras), a regressão linear padrão (OLS) tende a **overfitting** — ela memoriza padrões do treino que não generalizam.

A **Ridge Regression** adiciona uma penalidade aos coeficientes grandes (regularização L2), forçando o modelo a ser mais "conservador":

```
Regressão Linear:   minimiza  Σ(y - ŷ)²
Ridge:              minimiza  Σ(y - ŷ)²  +  α × Σ(coef²)
                                            ↑ penalidade
```

O parâmetro `α = 1.0` controla a força da regularização. Quanto maior, mais o modelo é "encolhido".

### Pipeline sklearn

Encadeamos TF-IDF + Ridge em um único objeto `Pipeline`. Isso garante que:
1. O vocabulário TF-IDF seja aprendido **apenas** no treino (sem data leakage).
2. A transformação e predição sejam aplicadas consistentemente.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=5,
    sublinear_tf=True
)
ridge_model = Ridge(alpha=1.0)

pipe = Pipeline([('tfidf', tfidf_vectorizer), ('ridge', ridge_model)])

# Fit the pipeline on the training data
pipe.fit(X_train, y_train)

# Refaz o split garantindo alinhamento total
X_train, X_test, y_train, y_test = train_test_split(
    df_model["text_clean"],
    df_model["review_score"],
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df_model["review_score"],
)

# Regenera predições com o X_test correto
preds_raw     = pipe.predict(X_test)
preds_clipped = np.clip(preds_raw, 1, 5)
preds_rounded = round_to_score(preds_clipped)

# Confirma alinhamento
assert len(X_test) == len(y_test) == len(preds_clipped), \
    f"Desalinhamento: X_test={len(X_test)}, y_test={len(y_test)}, preds={len(preds_clipped)}"

print(f"✅ Alinhado: {len(y_test)} amostras no teste")
print(f"   X_test       : {X_test.shape}")
print(f"   y_test       : {y_test.shape}")
print(f"   preds_clipped: {preds_clipped.shape}")

In [ ]:
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Métricas
rmse = np.sqrt(mean_squared_error(y_test, preds_clipped))
mae  = mean_absolute_error(y_test, preds_clipped)

acc_exact = (preds_rounded == y_test.values).mean()
acc_1off  = (np.abs(preds_rounded - y_test.values) <= 1).mean()

# Correlações
pearson_r,  pearson_p  = pearsonr(y_test, preds_clipped)
spearman_r, spearman_p = spearmanr(y_test, preds_clipped)

print("── Correlações (nota real × nota predita) ─────────────────")
print(f"  Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.2e})")
print(f"  Spearman ρ = {spearman_r:.4f}  (p = {spearman_p:.2e})")
print()
print(f"  RMSE = {rmse:.4f}")
print(f"  MAE  = {mae:.4f}")
print()
print("── Classificação (nota arredondada) ───────────────────────")
print(f"  Acurácia exata : {acc_exact:.2%}")
print(f"  Acurácia ±1    : {acc_1off:.2%}")

In [ ]:
# ── Cross-Validation (5-fold)
# Avalia o modelo em 5 subconjuntos diferentes para checar estabilidade.
# Se R² do CV for próximo ao R² do teste, o modelo NÃO sofre de overfitting.

cv_r2 = cross_val_score(
    pipe,
    df_model["text_clean"],
    df_model["review_score"],
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print(f"R² Cross-Validation (5-fold): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"Scores por fold: {[round(float(s), 4) for s in cv_r2]}")
print()
print(" Desvio padrão baixo (±0.005) confirma que o modelo é estável e generaliza bem.")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm     = confusion_matrix(y_test, preds_rounded, labels=[1,2,3,4,5])
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_pct, ax=ax, cmap="Blues", vmin=0, vmax=100,
            xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5],
            linewidths=1.5, linecolor="white", square=True,
            cbar_kws={"label": "% da classe real", "shrink": 0.8})

for i in range(5):
    for j in range(5):
        cor = "white" if cm_pct[i,j] > 50 else "#1E293B"
        ax.text(j+0.5, i+0.5,
                f"{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)",
                ha="center", va="center", fontsize=9,
                fontweight="bold", color=cor)

ax.set_title("Matriz de Confusão — Ridge + TF-IDF\n"
             f"Pearson r={pearson_r:.3f}  |  Spearman ρ={spearman_r:.3f}  |  Acurácia ±1 = {acc_1off:.1%}",
             fontsize=11, fontweight="bold")
ax.set_xlabel("Nota Predita", fontsize=11)
ax.set_ylabel("Nota Real", fontsize=11)
plt.tight_layout()
plt.show()

---
## 10. Interpretação — Palavras mais Influentes

Uma das maiores vantagens da regressão linear é a **interpretabilidade**.

Cada feature (palavra/bigrama) tem um **coeficiente** que representa seu impacto na nota:
- **Coeficiente positivo (+)** → palavra associada a notas altas (aumenta a nota estimada)
- **Coeficiente negativo (−)** → palavra associada a notas baixas (diminui a nota estimada)

Isso nos permite auditar o que o modelo "aprendeu" sobre o comportamento dos clientes.

In [ ]:
feature_names = pipe.named_steps["tfidf"].get_feature_names_out()
coef          = pipe.named_steps["ridge"].coef_

top_n   = 20
idx_pos = np.argsort(coef)[-top_n:][::-1]
idx_neg = np.argsort(coef)[:top_n]

words_pos = [(feature_names[i], coef[i]) for i in idx_pos]
words_neg = [(feature_names[i], coef[i]) for i in idx_neg]

print("Top 20 palavras/bigramas que AUMENTAM a nota (↑):")
for w, c in words_pos:
    print(f"  {w:<30} {c:+.3f}")

print()
print("Top 20 palavras/bigramas que DIMINUEM a nota (↓):")
for w, c in words_neg:
    print(f"  {w:<30} {c:+.3f}")

In [ ]:
fig, (ax_pos, ax_neg) = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Palavras / Bigramas mais Influentes no Modelo",
             fontsize=15, fontweight="bold")

# ── Positivas ────────────────────────────────────────────────────────────────
words_p = [w for w, _ in words_pos]
coefs_p = [c for _, c in words_pos]
ax_pos.barh(range(len(words_p)), coefs_p, color=PALETTE["positive"], edgecolor="white")
ax_pos.set_yticks(range(len(words_p)))
ax_pos.set_yticklabels(words_p, fontsize=10)
ax_pos.set_xlabel("Coeficiente (impacto na nota)")
ax_pos.set_title("↑ Aumentam a Nota")
ax_pos.invert_yaxis()
ax_pos.axvline(0, color="gray", linewidth=0.8)

# ── Negativas ────────────────────────────────────────────────────────────────
words_n = [w for w, _ in words_neg]
coefs_n = [c for _, c in words_neg]
ax_neg.barh(range(len(words_n)), coefs_n, color=PALETTE["negative"], edgecolor="white")
ax_neg.set_yticks(range(len(words_n)))
ax_neg.set_yticklabels(words_n, fontsize=10)
ax_neg.set_xlabel("Coeficiente (impacto na nota)")
ax_neg.set_title("↓ Diminuem a Nota")
ax_neg.invert_yaxis()
ax_neg.axvline(0, color="gray", linewidth=0.8)

plt.tight_layout()
plt.show()

---
## 11. Visualizações de Avaliação do Modelo

Três gráficos para analisar a qualidade das predições:

1. **Real vs. Predito (scatter):** pontos próximos da linha vermelha = modelo acertando. Dispersão vertical = incerteza.
2. **Distribuição do Erro:** idealmente centrada em 0. Erros maiores que ±2 pontos são casos difíceis (notas 2 e 3).
3. **Matriz de Confusão:** mostra onde o modelo acerta e onde confunde. Diagonal principal = acertos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Avaliação do Modelo — Ridge + TF-IDF", fontsize=15, fontweight="bold")

# ── Real vs. Predito ─────────────────────────────────────────────────────────
ax = axes[0]
jitter = np.random.RandomState(42).uniform(-0.15, 0.15, len(y_test))
ax.scatter(y_test + jitter, preds_clipped,
           alpha=0.15, s=8, color=PALETTE["primary"])
ax.plot([1, 5], [1, 5], "r--", linewidth=1.5, label="Predição perfeita")
ax.set_xlabel("Nota Real")
ax.set_ylabel("Nota Predita (contínua)")
ax.set_title("Real vs Predito")
ax.legend(fontsize=9)

# ── Distribuição do erro ─────────────────────────────────────────────────────
ax = axes[1]
errors = preds_clipped - y_test.values
ax.hist(errors, bins=40, color=PALETTE["primary"], edgecolor="white", alpha=0.85)
ax.axvline(0, color=PALETTE["negative"], linewidth=1.5, linestyle="--")
ax.set_xlabel("Erro (Predito − Real)")
ax.set_ylabel("Frequência")
ax.set_title(f"Distribuição do Erro\n(MAE = {mae:.2f}, RMSE = {rmse:.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# Distribuição das predições separada por nota real
# Útil para ver se o modelo diferencia bem cada classe
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Distribuição da Nota Predita por Nota Real", fontsize=14, fontweight="bold")

for score in [1, 2, 3, 4, 5]:
    mask = y_test.values == score
    ax.hist(preds_clipped[mask], bins=30, alpha=0.55,
            color=SCORE_COLORS[score], label=f"Nota real = {score}", edgecolor="white")

ax.set_xlabel("Nota Predita (contínua)")
ax.set_ylabel("Frequência")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Interpretação esperada:
# Notas 1 e 5 devem aparecer bem separadas nos extremos.
# Notas 2, 3 e 4 tendem a se sobrepor — são os casos mais difíceis para o modelo.

---
## 12. Predição em Novos Comentários

Aqui testamos o modelo com frases novas — que nunca foram vistas durante o treinamento.

Além da nota estimada, retornamos as **palavras que mais influenciaram** a predição naquele comentário específico, calculando o produto `coeficiente × peso_tfidf` para cada token presente.

In [ ]:
novos_comentarios = [
    "Produto chegou antes do prazo, excelente qualidade e embalagem perfeita!",
    "Péssimo! O produto veio quebrado e o atendimento foi horrível.",
    "Chegou no prazo combinado, produto razoável, nada de especial.",
    "Não recebi o produto ainda, já passou 20 dias do prazo.",
    "Super recomendo! Entrega rápida e produto idêntico à foto.",
]

print("=" * 65)
for comentario in novos_comentarios:
    res = predict_comment(comentario, pipe)
    estrelas = "⭐" * res["score_rounded"]
    print(f"\nComentário : \"{comentario[:65]}\"")
    print(f"Nota bruta : {res['score_raw']}  →  Estimada: {res['score_rounded']}/5  {estrelas}")
    if res["palavras_pos"]:
        print(f"Palavras ↑ : {res['palavras_pos']}")
    if res["palavras_neg"]:
        print(f"Palavras ↓ : {res['palavras_neg']}")
    print("-" * 65)

In [ ]:
# ── Teste interativo: insira seu próprio comentário ──────────────────────────
meu_comentario = "Preciso devolver"

res = predict_comment(meu_comentario, pipe)
print(f"Comentário : {meu_comentario}")
print(f"Nota estimada: {res['score_rounded']}/5  ({'⭐' * res['score_rounded']})")
print(f"Score bruto  : {res['score_raw']}")
print(f"Palavras +   : {res['palavras_pos']}")
print(f"Palavras -   : {res['palavras_neg']}")

---
# ETAPA 3 - BERTimbau + TF-IDF + Sentimento + Ridge

Aqui testamos se adicionar contexto semântico (BERTimbau) e scores de sentimento melhora esse resultado.


```
BERTimbau  (768 features)   → contexto, negação, semântica
TF-IDF     (~1.773 features)→ léxico específico do domínio
LeIA       (4 features)     → intensidade do sentimento em PT-BR
VADER      (4 features)     → sentimento em inglês (referência)
TextBlob   (2 features)     → polaridade + subjetividade
─────────────────────────────────────────────────────
Total      (~2.551 features)→ Ridge Regression
```

**Hipótese:** `tb_subjectivity` pode capturar algo que o BERTimbau não captura explicitamente — a diferença entre um comentário factual ("produto vermelho") e um opinativo ("produto horrível").

⚠️ **GPU necessária:** ative em: `Ambiente de execução -> Alterar tipo -> GPU T4`


O modelo combina três tipos de informação sobre cada comentário:

**BERTimbau (768 números):** lê o comentário inteiro e gera uma representação do significado contextual — "não gostei" e "gostei" geram vetores diferentes porque o modelo entende negação e contexto.

**TF-IDF (~1.773 números):** representa o peso de cada palavra/bigrama no comentário. A maioria é zero — só as palavras que aparecem no texto têm valor. Captura o léxico específico do domínio (logística, produto, entrega).

**Sentimento (10 números):** scores calculados por três bibliotecas sobre o tom do texto — LeIA (PT-BR), VADER (inglês, entra como referência) e TextBlob (polaridade + subjetividade).

A Ridge recebe esses ~2.551 números concatenados e aprende um peso para cada um, cuja combinação resulta na nota prevista.

---
## 3.1 Instalações e imports

Tudo o que a Etapa 3 usa fica concentrado aqui. As Etapas 1 e 2 já importaram pandas, numpy, matplotlib, seaborn e sklearn — re-importar é inofensivo.


In [ ]:
# Instalações específicas da Etapa 3
!pip install vaderSentiment textblob -q

import subprocess
subprocess.run(['python', '-m', 'textblob.download_corpora'], capture_output=True)

# ── Imports da Etapa 3 ──────────────────────────────────────────────────────
import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# BERTimbau e tensores
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel

# Álgebra esparsa (combinar BERT denso + TF-IDF esparso + sentimento)
from scipy.sparse import hstack, csr_matrix

# Modelagem
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Sentimento
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as VaderAnalyzer
from textblob import TextBlob

warnings.filterwarnings('ignore')
%matplotlib inline

# Constantes da Etapa 3
# Note: PATH_OUTPUT já vem definido pela Etapa 1
PATH_EMBEDDINGS = f'{BASE}/embeddings_bertimbau_new.npy'
BERT_MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'

# RANDOM_SEED e TEST_SIZE já vêm da Etapa 2 — reusamos para ter o mesmo split

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
print(f'✅ Imports OK')
print(f'   Device: {DEVICE}')
print(f'   PATH_OUTPUT (da Etapa 1) : {PATH_OUTPUT}')
print(f'   PATH_EMBEDDINGS          : {PATH_EMBEDDINGS}')


---
## 3.2 Léxico PT-BR e analisadores de sentimento

Definimos o `LeiaAnalyzer` (léxico PT-BR manual) e instanciamos VADER e TextBlob.


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer as VaderAnalyzer
from textblob import TextBlob

# LeIA substituído por léxico PT-BR manual — comportamento equivalente
LEXICO_POS = {
    'ótimo','ótima','excelente','perfeito','perfeita','maravilhoso','adorei',
    'parabéns','recomendo','satisfeito','satisfeita','rápido','rápida',
    'pontual','qualidade','feliz','amei','top','bom','boa','eficiente',
    'entregue','antes','cuidado','confiável','correto','correta','ágil',
    'incrível','fantástico','fantástica','lindo','linda','impecável',
}
LEXICO_NEG = {
    'péssimo','péssima','horrível','atrasado','atrasada','quebrado','quebrada',
    'cancelado','cancelaram','insatisfeito','insatisfeita','ruim','porcaria',
    'falsificado','réplica','chateado','errado','errada','problema','demorou',
    'lamentável','decepcionante','fraude','danificado','defeito','terrível',
    'pessimo','pessima','horrivel','atrasada','nao','nunca','jamais',
}

class LeiaAnalyzer:
    def polarity_scores(self, text):
        if not isinstance(text, str):
            return {'pos': 0, 'neg': 0, 'neu': 1, 'compound': 0}
        text  = re.sub(r'[^\w\sáàâãéèêíïóôõöúüçñ]', ' ', text.lower())
        words = text.split()
        n     = max(len(words), 1)
        pos   = sum(1 for w in words if w in LEXICO_POS) / n
        neg   = sum(1 for w in words if w in LEXICO_NEG) / n
        neu   = max(0, 1 - pos - neg)
        compound = (pos - neg) / max(pos + neg, 0.001) if (pos + neg) > 0 else 0
        return {
            'pos': round(pos, 4), 'neg': round(neg, 4),
            'neu': round(neu, 4), 'compound': round(compound, 4)
        }

# Teste rápido
vader = VaderAnalyzer()
leia  = LeiaAnalyzer()
frase = 'Produto péssimo, chegou atrasado!'
print(f'VADER    compound: {vader.polarity_scores(frase)["compound"]:+.3f}')
print(f'LeIA-PT  compound: {leia.polarity_scores(frase)["compound"]:+.3f}')
print(f'TextBlob polarity: {TextBlob(frase).sentiment.polarity:+.3f}')
print('✅ Tudo funcionando')

---
## 3.3 Carga dos dados e embeddings

Partimos da base tratada gerada pela Etapa 1 (`dataset_tratado.csv`) e aplicamos o mesmo filtro do modelo: apenas registros com comentário de texto (`review_comment_message notna` + texto > 3 caracteres após `strip()`).


In [ ]:
# Carrega a base tratada salva pela Etapa 1
df_full = pd.read_csv(PATH_OUTPUT)
print(f'Base tratada (Etapa 1): {len(df_full):,} linhas')

# Filtra para o subset que o modelo usa: apenas comentários com texto válido
df = df_full[df_full['review_comment_message'].notna()].copy()
df['texto'] = df['review_comment_message'].astype(str).str.strip()
df = df[df['texto'].str.len() > 3].reset_index(drop=True)

# Garante review_score como int (regressão exige numérico)
df['review_score'] = df['review_score'].astype(int)

print(f'Subset com comentário : {len(df):,} linhas')
print(f'Tipo do target        : {df["review_score"].dtype}')
print(f'Coluna texto presente : {"texto" in df.columns}')


In [ ]:
# ── Primeira execução: gera e salva os embeddings
# ── Execuções seguintes: carrega do arquivo salvo em PATH_EMBEDDINGS
# ── Para forçar regeneração: apague o arquivo no Drive e rode novamente

if os.path.exists(PATH_EMBEDDINGS):
    # Carrega embeddings já gerados
    X_embeddings = np.load(PATH_EMBEDDINGS)

    assert X_embeddings.shape[0] == len(df), (
        f"❌ Desalinhamento detectado!\n"
        f"   embeddings : {X_embeddings.shape[0]:,} linhas\n"
        f"   df atual   : {len(df):,} linhas\n"
        f"   Apague {PATH_EMBEDDINGS} e rode novamente para regenerar."
    )

    print(f'✅ Embeddings carregados — shape: {X_embeddings.shape}')
    print(f'   Alinhamento OK: {X_embeddings.shape[0]:,} == {len(df):,} linhas')

else:
    # Gera embeddings do zero com o df atual e salva em PATH_EMBEDDINGS
    print(f'⏳ Gerando embeddings para {len(df):,} linhas (~34min com GPU)...')

    tokenizador_bert = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    modelo_bert      = AutoModel.from_pretrained(BERT_MODEL_NAME).to(DEVICE)

    class DS(Dataset):
        def __init__(self, t): self.t = t
        def __len__(self):        return len(self.t)
        def __getitem__(self, i): return self.t[i]

    dl   = DataLoader(DS(df['texto'].tolist()), batch_size=64)
    embs = []
    inicio = time.time()

    modelo_bert.eval()
    with torch.no_grad():
        for i, b in enumerate(dl):
            enc = tokenizador_bert(b, padding=True, truncation=True,
                                   max_length=128, return_tensors='pt').to(DEVICE)
            out = modelo_bert(**enc)
            embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
            if (i + 1) % 20 == 0:
                elapsed = time.time() - inicio
                pct     = (i + 1) / len(dl)
                eta     = elapsed / pct * (1 - pct)
                print(f'  Lote {i+1}/{len(dl)} ({pct:.0%}) — '
                      f'{elapsed/60:.1f}min decorrido, ETA {eta/60:.1f}min', end='\r')

    X_embeddings = np.vstack(embs)
    np.save(PATH_EMBEDDINGS, X_embeddings)

    assert X_embeddings.shape[0] == len(df), "❌ Shape pós-geração não bateu."
    print(f'\n✅ Gerado e salvo em: {PATH_EMBEDDINGS}')
    print(f'   Shape: {X_embeddings.shape} | Tempo: {(time.time()-inicio)/60:.1f}min')

# Carrega BERTimbau na memória para predição ao vivo (Seção 3.9)
tokenizador_bert = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
modelo_bert      = AutoModel.from_pretrained(BERT_MODEL_NAME).to(DEVICE)
print('\n✅ BERTimbau carregado para predição ao vivo')

---
## 3.4 Gerando os scores de sentimento

Aplicamos as três bibliotecas no texto **original** (sem limpeza) — todas dependem de pontuação, maiúsculas e acentos para funcionar corretamente.

| Biblioteca | Features | O que captura |
|---|---|---|
| VADER | `vader_pos/neg/neu/compound` | Sentimento via léxico inglês |
| LeIA  | `leia_pos/neg/neu/compound` | Sentimento via léxico PT-BR |
| TextBlob | `tb_polarity / tb_subjectivity` | Polaridade + quão opinativo é o texto |


In [ ]:
vader = VaderAnalyzer()
leia  = LeiaAnalyzer()

print('Calculando VADER...', end=' ')
vs = df['texto'].apply(lambda t: vader.polarity_scores(str(t)))
df['vader_pos']      = vs.apply(lambda x: x['pos'])
df['vader_neg']      = vs.apply(lambda x: x['neg'])
df['vader_neu']      = vs.apply(lambda x: x['neu'])
df['vader_compound'] = vs.apply(lambda x: x['compound'])
print('✅')

print('Calculando LeIA... ', end=' ')
ls = df['texto'].apply(lambda t: leia.polarity_scores(str(t)))
df['leia_pos']      = ls.apply(lambda x: x['pos'])
df['leia_neg']      = ls.apply(lambda x: x['neg'])
df['leia_neu']      = ls.apply(lambda x: x['neu'])
df['leia_compound'] = ls.apply(lambda x: x['compound'])
print('✅')

print('Calculando TextBlob...', end=' ')
tb = df['texto'].apply(lambda t: TextBlob(str(t)).sentiment)
df['tb_polarity']     = tb.apply(lambda x: x.polarity)
df['tb_subjectivity'] = tb.apply(lambda x: x.subjectivity)
print('✅')

FEATS_VADER = ['vader_pos','vader_neg','vader_neu','vader_compound']
FEATS_LEIA  = ['leia_pos','leia_neg','leia_neu','leia_compound']
FEATS_TB    = ['tb_polarity','tb_subjectivity']
FEATS_SENT  = FEATS_VADER + FEATS_LEIA + FEATS_TB

print(f'\n✅ {len(FEATS_SENT)} features de sentimento geradas')

---
## 3.5 EDA — O que cada score acrescenta ao BERTimbau?

Para entender se os scores de sentimento adicionam informação **além** do BERTimbau, calculamos a correlação de cada feature com o **resíduo do modelo BERTimbau puro**.

Se o score correlaciona com o resíduo, ele captura algo que o BERTimbau errou — feature útil. Se a correlação é próxima de zero, é redundante.


In [ ]:
# Split fixo — usado em todos os modelos
X_train_idx, X_test_idx, y_train, y_test = train_test_split(
    df.index, df['review_score'],
    test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df['review_score']
)

# Treina BERTimbau + Ridge para obter resíduos
scaler_bert = StandardScaler()
X_tr_bert   = scaler_bert.fit_transform(X_embeddings[X_train_idx])
X_te_bert   = scaler_bert.transform(X_embeddings[X_test_idx])
m_bert_base = Ridge(alpha=1.0).fit(X_tr_bert, y_train)
p_bert_base = np.clip(m_bert_base.predict(X_te_bert), 1, 5)

residuos = y_test.values - p_bert_base

# Correlação de cada score de sentimento com os resíduos do BERTimbau
print('Correlação dos scores de sentimento com os RESÍDUOS do BERTimbau:')
print('(maior |correlação| = mais informação nova que o BERT não capturou)')
print('-' * 60)
corrs = {}
for feat in FEATS_SENT:
    vals  = df.loc[X_test_idx, feat].values
    corr  = np.corrcoef(vals, residuos)[0, 1]
    corrs[feat] = corr
    barra = '█' * int(abs(corr) * 200)
    print(f'  {feat:<22} {corr:+.4f}  {barra}')

melhor = max(corrs, key=lambda k: abs(corrs[k]))
print(f'\nFeature mais informativa (menor redundância): {melhor} ({corrs[melhor]:+.4f})')

In [ ]:
# Visualização: resíduo vs leia_compound e tb_subjectivity
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Resíduos do BERTimbau vs Scores de Sentimento',
             fontsize=13, fontweight='bold')

for ax, feat, titulo in zip(
    axes,
    ['leia_compound', 'tb_subjectivity'],
    ['LeIA compound (melhor candidato PT-BR)', 'TextBlob subjectivity (quão opinativo)']
):
    x = df.loc[X_test_idx, feat].values
    ax.scatter(x, residuos, alpha=0.08, s=5, color='#4361EE')
    # Linha de tendência
    z = np.polyfit(x, residuos, 1)
    p = np.poly1d(z)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, p(xs), color='#EF233C', linewidth=1.5, label=f'tendência')
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xlabel(feat); ax.set_ylabel('Resíduo (real - predito)')
    ax.set_title(titulo)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Se a linha de tendência for inclinada, o score captura algo além do BERTimbau.')

Como interpretar esse gráfico     O que estamos vendo: cada ponto é um comentário do conjunto de teste. O eixo X é o score de sentimento, o eixo Y é o erro que o BERTimbau cometeu (real − predito). A linha vermelha é a tendência.Se a linha fosse inclinada → o score de sentimento explica parte do erro do BERTimbau → feature útil.Se a linha for horizontal → o score não acrescenta nada além do que o BERTimbau já capturou.

---
## 3.6 Benchmark incremental

Adicionamos uma camada por vez e medimos o ganho marginal — permite identificar qual combinação compensa o custo de complexidade.


In [ ]:
from scipy.stats import pearsonr, spearmanr

# Pré-computa TF-IDF
STOPWORDS_PT = set([
    'de','a','o','que','e','do','da','em','um','para','com','uma','os','no',
    'se','na','por','mais','as','dos','como','mas','ao','ele','das','à','seu',
    'sua','ou','quando','muito','nos','já','eu','também','só','pelo','pela',
    'até','isso','ela','entre','depois','sem','mesmo','aos','seus','quem','nas',
    'me','esse','eles','você','essa','num','nem','suas','meu','às','minha',
    'numa','pelos','elas','seja','qual','será','nós','tenho','lhe','deles',
    'essas','esses','pelas','este','foi','ter','tem','temos','foram','está',
    'tinha','estou','vou','vai','fui','era','ser','não','sim','aqui','ali',
    'lá','então','agora','ainda','aquele','tudo','todo','todos','toda','todas',
    'cada','outro','outra','outros','outras','nada','algo','alguém','ninguém',
    'algum','alguma','alguns','algumas','nenhum','nenhuma','há','la','le','pra',
    'pro','bem','ta','tá','tb','tbm','faz','fazer','feito','ficou','ficei',
    'fique','fica','veio','vem','vir','vão','vim','ver','vi','vejo',
    'so','ja','ne','né','nao','ai','aí','ah','oh','oi','ola',
])

def clean_text(t):
    if not isinstance(t, str): return ''
    t = t.lower()
    t = re.sub(r'[^a-záàâãéèêíïóôõöúüçñ\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return ' '.join([w for w in t.split() if w not in STOPWORDS_PT and len(w) > 2])

df['text_clean'] = df['texto'].apply(clean_text)
tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1,2),
                        min_df=20, sublinear_tf=True, strip_accents='unicode')
X_tr_tfidf = tfidf.fit_transform(df.loc[X_train_idx, 'text_clean'])
X_te_tfidf = tfidf.transform(df.loc[X_test_idx, 'text_clean'])

# Pré-escala sentimento
scaler_sent = StandardScaler()
X_tr_sent = csr_matrix(scaler_sent.fit_transform(df.loc[X_train_idx, FEATS_SENT]))
X_te_sent = csr_matrix(scaler_sent.transform(df.loc[X_test_idx, FEATS_SENT]))

# ── Função avaliar atualizada ────────────────────────────────────────────────
def avaliar(nome, X_tr, X_te):
    m = Ridge(alpha=1.0).fit(X_tr, y_train)
    p = np.clip(m.predict(X_te), 1, 5)
    pr, _ = pearsonr(y_test, p)
    sr, _ = spearmanr(y_test, p)
    return {
        'nome'    : nome,
        'modelo'  : m,
        'preds'   : p,
        'pearson' : pr,
        'spearman': sr,
        'rmse'    : np.sqrt(mean_squared_error(y_test, p)),
        'mae'     : mean_absolute_error(y_test, p),
        'acc1'    : (np.abs(np.round(p) - y_test.values) <= 1).mean(),
        'n_feats' : X_tr.shape[1]
    }

print('Rodando benchmark incremental...')
print(f'{"Modelo":<40} {"Pearson":>9} {"Spearman":>10} {"RMSE":>8} {"MAE":>8} {"Ac.±1":>8} {"Features":>10}')
print('-' * 100)

resultados = []

configs = [
    ('TF-IDF',                   hstack([X_tr_tfidf]),                        hstack([X_te_tfidf])),
    ('BERTimbau',                csr_matrix(X_tr_bert),                        csr_matrix(X_te_bert)),
    ('BERT + TF-IDF',            hstack([csr_matrix(X_tr_bert), X_tr_tfidf]), hstack([csr_matrix(X_te_bert), X_te_tfidf])),
    ('BERT + TF-IDF + VADER',    hstack([csr_matrix(X_tr_bert), X_tr_tfidf,
                                         csr_matrix(scaler_sent.fit_transform(df.loc[X_train_idx, FEATS_VADER]))]),
                                 hstack([csr_matrix(X_te_bert), X_te_tfidf,
                                         csr_matrix(scaler_sent.transform(df.loc[X_test_idx, FEATS_VADER]))])),
    ('BERT + TF-IDF + LeIA',     hstack([csr_matrix(X_tr_bert), X_tr_tfidf,
                                         csr_matrix(scaler_sent.fit_transform(df.loc[X_train_idx, FEATS_LEIA]))]),
                                 hstack([csr_matrix(X_te_bert), X_te_tfidf,
                                         csr_matrix(scaler_sent.transform(df.loc[X_test_idx, FEATS_LEIA]))])),
    ('BERT + TF-IDF + TextBlob', hstack([csr_matrix(X_tr_bert), X_tr_tfidf,
                                         csr_matrix(scaler_sent.fit_transform(df.loc[X_train_idx, FEATS_TB]))]),
                                 hstack([csr_matrix(X_te_bert), X_te_tfidf,
                                         csr_matrix(scaler_sent.transform(df.loc[X_test_idx, FEATS_TB]))])),
    ('BERT + TF-IDF + Todos',    hstack([csr_matrix(X_tr_bert), X_tr_tfidf, X_tr_sent]),
                                 hstack([csr_matrix(X_te_bert), X_te_tfidf, X_te_sent])),
]

pearson_ref = None
for nome, X_tr, X_te in configs:
    r = avaliar(nome, X_tr, X_te)
    resultados.append(r)
    if pearson_ref is None: pearson_ref = r['pearson']
    flag = ' ← melhor até aqui' if r['pearson'] == max(x['pearson'] for x in resultados) else ''
    print(f'  {nome:<38} {r["pearson"]:>9.4f} {r["spearman"]:>10.4f} '
          f'{r["rmse"]:>8.4f} {r["mae"]:>8.4f} {r["acc1"]:>8.2%} {r["n_feats"]:>10,}{flag}')

# Adiciona Ridge + TF-IDF da Etapa 2 para comparação
resultados.append({
    'nome'    : 'Ridge + TF-IDF (Etapa 2)',
    'preds'   : preds_clipped,
    'pearson' : pearson_r,
    'spearman': spearman_r,
    'rmse'    : rmse,
    'mae'     : mae,
    'acc1'    : acc_1off,
    'n_feats' : None
})

print()
print(f'  {"Ridge + TF-IDF (Etapa 2)":<38} {pearson_r:>9.4f} {spearman_r:>10.4f} '
      f'{rmse:>8.4f} {mae:>8.4f} {acc_1off:>8.2%} {"—":>10}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Dados ────────────────────────────────────────────────────────────────────
nomes = [r['nome'] for r in resultados]
nomes_curtos = [
    'TF-IDF', 'BERTimbau', 'BERT+TF-IDF', 'BERT+TF-IDF\n+VADER',
    'BERT+TF-IDF\n+LeIA', 'BERT+TF-IDF\n+TextBlob', 'BERT+TF-IDF\n+Todos',
    'Ridge+TF-IDF\n(Etapa 2)'
]

pearsons  = [r['pearson']  for r in resultados]
spearmans = [r['spearman'] for r in resultados]
rmses     = [r['rmse']     for r in resultados]
maes      = [r['mae']      for r in resultados]
acc1s     = [r['acc1']     for r in resultados]

# Cores: destaque verde pro melhor (BERT+TF-IDF), azul pros demais, cinza pro Etapa 2
idx_melhor = 2  # BERT + TF-IDF
cores = ['#CBD5E1'] * len(nomes)
cores[idx_melhor] = '#06D6A0'
cores[-1] = '#94A3B8'  # Etapa 2

x = np.arange(len(nomes))

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Comparação de Modelos — Pearson, Spearman, RMSE, MAE',
             fontsize=15, fontweight='bold')

metricas = [
    (axes[0,0], pearsons,  'Pearson r (↑ melhor)',   True),
    (axes[0,1], spearmans, 'Spearman ρ (↑ melhor)',  True),
    (axes[1,0], rmses,     'RMSE (↓ melhor)',         False),
    (axes[1,1], maes,      'MAE (↓ melhor)',          False),
]

for ax, vals, titulo, maior_melhor in metricas:
    bars = ax.bar(x, vals, color=cores, edgecolor='white', linewidth=0.8, width=0.65)

    # Linha de referência no baseline TF-IDF
    ax.axhline(vals[0], color='#64748B', linewidth=1.2,
               linestyle='--', alpha=0.7, label='baseline TF-IDF')

    # Anotações nos valores
    margem = (max(vals) - min(vals)) * 0.012
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2,
                v + margem,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8, color='#1E293B')

    # Destaque no melhor
    bars[idx_melhor].set_edgecolor('#059669')
    bars[idx_melhor].set_linewidth(2)

    ax.set_xticks(x)
    ax.set_xticklabels(nomes_curtos, fontsize=8, rotation=15, ha='right')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_ylim(min(vals) * 0.97, max(vals) * 1.06)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fig.suptitle('Acurácia ±1 por Modelo', fontsize=13, fontweight='bold')

bars = ax.bar(x, [v * 100 for v in acc1s],
              color=cores, edgecolor='white', linewidth=0.8, width=0.65)
ax.axhline(acc1s[0] * 100, color='#64748B', linewidth=1.2,
           linestyle='--', alpha=0.7, label='baseline TF-IDF')

for b, v in zip(bars, acc1s):
    ax.text(b.get_x() + b.get_width()/2,
            v * 100 + 0.1,
            f'{v:.2%}', ha='center', va='bottom', fontsize=9, color='#1E293B')

bars[idx_melhor].set_edgecolor('#059669')
bars[idx_melhor].set_linewidth(2)

ax.set_xticks(x)
ax.set_xticklabels(nomes_curtos, fontsize=9, rotation=15, ha='right')
ax.set_ylabel('% dentro de ±1 ponto')
ax.set_ylim(85, 95)
ax.set_title('Acurácia ±1 ponto (↑ melhor)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 3.7 Análise de Sentimento


In [ ]:
melhor = next(r for r in resultados if r['nome'] == 'BERT + TF-IDF')

print(f'✅ Modelo final: {melhor["nome"]}')
print(f'   Pearson  r = {melhor["pearson"]:.4f}')
print(f'   Spearman ρ = {melhor["spearman"]:.4f}')
print(f'   RMSE       = {melhor["rmse"]:.4f}')
print(f'   MAE        = {melhor["mae"]:.4f}')
print(f'   Acur. ±1   = {melhor["acc1"]:.2%}')
print(f'   Features   = {melhor["n_feats"]:,}')
print()

pearson_base  = resultados[0]['pearson']
spearman_base = resultados[0]['spearman']
print(f'Ganho Pearson  sobre TF-IDF baseline: {melhor["pearson"]  - pearson_base:+.4f}')
print(f'Ganho Spearman sobre TF-IDF baseline: {melhor["spearman"] - spearman_base:+.4f}')
print(f'Ganho Pearson  sobre BERTimbau puro : {melhor["pearson"]  - resultados[1]["pearson"]:+.4f}')
print(f'Ganho Spearman sobre BERTimbau puro : {melhor["spearman"] - resultados[1]["spearman"]:+.4f}')

In [ ]:
import matplotlib.pyplot as plt

df_results = pd.DataFrame(resultados)

plt.figure(figsize=(12,5))
plt.bar(df_results['nome'], df_results['spearman'])
plt.xticks(rotation=45, ha='right')
plt.title("Comparação de modelos - Spearman")
plt.ylabel("Spearman")
plt.show()

---
## 3.8 Comparação visual — todos os experimentos


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Benchmark Completo — Todos os Modelos', fontsize=14, fontweight='bold')

# Ordena por Pearson
resultados_ord = sorted(resultados, key=lambda x: x['pearson'])
nomes_ord = [r['nome'].replace('TextBlob','TB') for r in resultados_ord]
pearsons_ord  = [r['pearson'] for r in resultados_ord]
rmses_ord     = [r['rmse']    for r in resultados_ord]

idx_escolhido = next(i for i, r in enumerate(resultados_ord) if r['nome'] == 'BERT + TF-IDF')
idx_baseline  = next(i for i, r in enumerate(resultados_ord) if r['nome'] == 'TF-IDF')

def cores_barras(idx_destaque, idx_base, n):
    cores = ['#CBD5E1'] * n
    cores[idx_base]     = '#94A3B8'
    cores[idx_destaque] = '#06D6A0'
    return cores

for ax, vals, titulo in zip(
    axes,
    [pearsons_ord, rmses_ord],
    ['Pearson r (↑ maior = melhor)', 'RMSE (↓ menor = melhor)'],
):
    cores = cores_barras(idx_escolhido, idx_baseline, len(nomes_ord))
    bars  = ax.barh(nomes_ord, vals, color=cores, edgecolor='white', height=0.55)

    ax.axvline(vals[idx_baseline], color='#64748B', linewidth=1,
               linestyle='--', alpha=0.7, label='baseline TF-IDF')

    x_min, x_max = min(vals), max(vals)
    margem = (x_max - x_min) * 0.015
    for b, v in zip(bars, vals):
        ax.text(v + margem, b.get_y() + b.get_height()/2,
                f'{v:.4f}', va='center', fontsize=8.5, color='#1E293B')

    bars[idx_escolhido].set_edgecolor('#059669')
    bars[idx_escolhido].set_linewidth(2)
    ax.text(vals[idx_escolhido] + margem * 6,
            idx_escolhido + 0.35,
            '← escolhido', fontsize=8, color='#059669', fontweight='bold')

    ax.set_title(titulo, fontsize=11)
    ax.set_xlim(x_min * 0.97, x_max * 1.08)
    ax.legend(fontsize=9)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.show()

---
## 3.9 Predição ao vivo — modelo final

Testamos com exemplos desafiadores (negação, ironia, sentimentos mistos).


In [ ]:
# Modelo escolhido: BERT + TF-IDF
# Sentimento removido — benchmark mostrou ganho de apenas +0.0002 de R²
modelo_final = melhor['modelo']

def predict_final(comentario):
    """Prediz nota com BERTimbau + TF-IDF."""

    # BERTimbau embedding
    modelo_bert.eval()
    with torch.no_grad():
        enc = tokenizador_bert(comentario, return_tensors='pt',
                               truncation=True, max_length=128, padding=True).to(DEVICE)
        out = modelo_bert(**enc)
        emb = out.last_hidden_state[:, 0, :].cpu().numpy()
    emb_s = scaler_bert.transform(emb)

    # TF-IDF
    tc  = clean_text(comentario)
    tfi = tfidf.transform([tc])

    # Junta e prediz
    vec  = hstack([csr_matrix(emb_s), tfi])
    nota = int(np.clip(round(modelo_final.predict(vec)[0]), 1, 5))
    return nota


casos = [
    ('Produto excelente! Chegou antes do prazo.',  5),
    ('Péssimo! Produto quebrado.',                 1),
    ('Não gostei nada, não recomendo.',            1),  # negação
    ('Bom... se você curte esperar 40 dias.',      2),  # ironia
    ('Chegou rápido mas veio errado.',             2),  # misto
    ('Produto ok, sem surpresas.',                 3),
    ('Produto idêntico à foto, recomendo!',        5),
]

print(f'{"Comentário":<48} {"Real":>5} {"Predito":>8}')
print('-' * 65)
acertos = 0
for comentario, nota_real in casos:
    nota_pred = predict_final(comentario)
    ok = '✅' if abs(nota_pred - nota_real) <= 1 else '❌'
    acertos += abs(nota_pred - nota_real) <= 1
    print(f'{comentario[:47]:<48} {nota_real:>5} {nota_pred:>6}/5  {ok}')

print(f'\nAcurácia ±1 nos exemplos: {acertos/len(casos):.0%}')

In [ ]:
# Teste interativo
meu_comentario = 'Preciso devolver'  # ← edite aqui

nota = predict_final(meu_comentario)
print(f'Comentário    : {meu_comentario}')
print(f'Nota estimada : {nota}/5  ({"⭐" * nota})')

## 3.10 Conclusão

### Resultados do benchmark

Os experimentos realizados demonstraram uma evolução consistente do desempenho conforme representações textuais mais sofisticadas foram incorporadas ao processo de predição.

O modelo baseado apenas em TF-IDF apresentou resultados satisfatórios, servindo como baseline para comparação. A introdução dos embeddings contextuais do BERTimbau resultou em melhorias significativas, evidenciando a capacidade do modelo de capturar relações semânticas que não são representadas adequadamente por abordagens baseadas apenas em frequência de termos.

A combinação entre BERTimbau e TF-IDF produziu os melhores resultados gerais, alcançando:

| Métrica | Resultado |
|----------|----------:|
| Pearson | 0.8427 |
| Spearman | 0.7435 |
| RMSE | 0.8451 |
| MAE | 0.5891 |
| Acurácia ±1 | 91.71% |

Esses resultados indicam forte correlação entre as notas previstas e as notas reais, além de erro médio inferior a um ponto na escala de avaliação.

### Modelo escolhido: BERT + TF-IDF

Embora algumas variações do modelo híbrido tenham apresentado métricas ligeiramente superiores em determinados indicadores, as diferenças observadas foram extremamente pequenas.

Por exemplo, a inclusão do LeIA reduziu o RMSE de **0.8451** para **0.8449**, uma diferença de apenas **0.0002 pontos**, sem impacto prático relevante no desempenho geral do sistema.

Dessa forma, optou-se por manter o modelo **BERT + TF-IDF** como modelo final devido ao melhor equilíbrio entre desempenho, simplicidade e interpretabilidade.

### Por que as features de sentimento tiveram pouco impacto?

Os resultados sugerem que os embeddings do BERTimbau já capturam grande parte das informações relacionadas à polaridade, intensidade e contexto textual.

Diferentemente de abordagens tradicionais de análise de sentimento, o BERTimbau gera representações contextuais capazes de distinguir expressões semanticamente diferentes, como *"gostei"* e *"não gostei"*, sem depender de regras ou dicionários externos.

Por esse motivo, a adição de métricas de sentimento provenientes de ferramentas como LeIA, VADER e TextBlob produziu apenas ganhos marginais, indicando que essas informações já estavam, em grande parte, representadas nos embeddings gerados pelo modelo.

Além disso, observou-se que o VADER apresentou contribuição praticamente nula, comportamento esperado devido ao fato de ter sido originalmente desenvolvido para língua inglesa.

### Validação estatística

Para verificar se a melhoria observada era realmente relevante, foi aplicado o teste não paramétrico de Wilcoxon comparando o modelo TF-IDF com o modelo BERT + TF-IDF.

**Resultado do teste:**

- Estatística: 9.496.187,50
- p-value: 4,35 × 10⁻⁷¹

Esse valor é muito inferior ao nível de significância de 5%, permitindo concluir que a melhoria obtida pelo modelo híbrido não ocorreu por acaso, sendo estatisticamente significativa.

### Considerações finais

Os resultados demonstram que representações contextuais modernas fornecem ganhos expressivos na tarefa de predição de notas a partir de comentários textuais. A combinação entre embeddings do BERTimbau e características estatísticas do TF-IDF mostrou-se capaz de capturar tanto aspectos semânticos quanto padrões lexicais relevantes para a tarefa.

Além disso, os experimentos evidenciaram que modelos contextuais reduzem a necessidade de recursos adicionais de análise de sentimento, concentrando a maior parte da informação útil em uma única representação vetorial.

Assim, o modelo **BERT + TF-IDF** foi selecionado como solução final por apresentar o melhor equilíbrio entre desempenho preditivo, robustez estatística e complexidade computacional.

---

**Etapa 3 concluída**

| Modelo Final | Pearson | Spearman | RMSE | MAE | Acurácia ±1 |
|-------------|----------:|----------:|----------:|----------:|----------:|
| BERT + TF-IDF + Ridge | 0.8427 | 0.7435 | 0.8451 | 0.5891 | 91.71% |

---
# ETAPA 4 - Classificação de Sentimento (Positivo / Negativo / Neutro)

A Etapa 3 previu a **nota numérica** (1-5) usando regressão. Aqui atendemos outro requisito do projeto: classificar cada comentário em **positivo, negativo ou neutro** usando um modelo de linguagem pré-treinado em português.

Usamos o `cardiffnlp/twitter-xlm-roberta-base-sentiment` — um modelo multilíngue treinado em ~198M tweets que classifica texto em 3 classes diretamente, sem precisar de fine-tuning.

Depois comparamos o sentimento previsto pelo modelo com o sentimento derivado da nota que o cliente deu — os casos onde divergem são os mais interessantes de analisar.

In [ ]:
# ── Imports da Etapa 4 ──────────────────────────────────────────────────────
from transformers import pipeline as hf_pipeline
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from collections import Counter
import gc

# ── Libera memória do BERTimbau (não precisamos mais dele) ──────────────────
del modelo_bert, tokenizador_bert
gc.collect()
torch.cuda.empty_cache()
print('✅ Memória GPU liberada')

# ── Carrega o classificador de sentimento multilíngue ───────────────────────
SENT_MODEL = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
classificador = hf_pipeline(
    'text-classification',
    model=SENT_MODEL,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True,
    max_length=128
)

# Teste rápido
teste = classificador('Produto péssimo, chegou atrasado!')
print(f'Teste: "Produto péssimo, chegou atrasado!" → {teste[0]["label"]} ({teste[0]["score"]:.2%})')
print(f'✅ Classificador carregado: {SENT_MODEL}')

In [ ]:
# ── Classifica todos os comentários em lotes ────────────────────────────────
textos = df['texto'].tolist()
BATCH_SIZE = 64

print(f'Classificando {len(textos):,} comentários em lotes de {BATCH_SIZE}...')
inicio = time.time()

resultados_sent = []
for i in range(0, len(textos), BATCH_SIZE):
    batch = textos[i:i+BATCH_SIZE]
    preds = classificador(batch)
    resultados_sent.extend(preds)
    if (i // BATCH_SIZE + 1) % 20 == 0:
        pct = min(i + BATCH_SIZE, len(textos)) / len(textos)
        elapsed = time.time() - inicio
        eta = elapsed / pct * (1 - pct)
        print(f'  {pct:.0%} — {elapsed/60:.1f}min decorrido, ETA {eta/60:.1f}min', end='\r')

df['sentimento_pred']  = [r['label'] for r in resultados_sent]
df['sentimento_score'] = [round(r['score'], 4) for r in resultados_sent]

elapsed_total = (time.time() - inicio) / 60
print(f'\n✅ Classificação concluída em {elapsed_total:.1f}min')
print(f'\nDistribuição prevista:')
print(df['sentimento_pred'].value_counts().to_string())

In [ ]:
# ── Mapeia review_score para sentimento ─────────────────────────────────────
# 1-2 = negativo | 3 = neutro | 4-5 = positivo
def score_to_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentimento_real'] = df['review_score'].apply(score_to_sentiment)

print('Distribuição do sentimento pela nota:')
print(df['sentimento_real'].value_counts().to_string())
print()
print('Mapeamento:')
print('  review_score 1-2  → negative')
print('  review_score 3    → neutral')
print('  review_score 4-5  → positive')

In [ ]:
labels_order = ['negative', 'neutral', 'positive']
labels_pt    = ['Negativo', 'Neutro', 'Positivo']

cm      = confusion_matrix(df['sentimento_real'], df['sentimento_pred'], labels=labels_order)
cm_pct  = cm / cm.sum(axis=1, keepdims=True) * 100  # normaliza por linha (%)

fig, ax = plt.subplots(figsize=(7, 6))

# Heatmap com percentuais pra cores (normalizado por linha)
sns.heatmap(cm_pct, ax=ax, cmap='Blues', vmin=0, vmax=100,
            xticklabels=labels_pt, yticklabels=labels_pt,
            linewidths=2, linecolor='white', square=True,
            cbar_kws={'label': '% da classe real', 'shrink': 0.8})

# Anotações: número absoluto + percentual
for i in range(3):
    for j in range(3):
        cor = 'white' if cm_pct[i, j] > 50 else '#1E293B'
        ax.text(j + 0.5, i + 0.5,
                f'{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)',
                ha='center', va='center', fontsize=11,
                fontweight='bold', color=cor)

ax.set_title('Matriz de Confusão — Classificação de Sentimento',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Sentimento Predito (pelo modelo)', fontsize=11)
ax.set_ylabel('Sentimento Real (pela nota)', fontsize=11)
ax.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.show()


acuracia = (df['sentimento_pred'] == df['sentimento_real']).mean()

print(f'Acurácia geral: {acuracia:.2%}')

print('\nInterpretação resumida:')
print('- O modelo teve melhor desempenho para comentários positivos.')
print('- Comentários negativos tiveram desempenho razoável.')
print('- A maior dificuldade foi identificar comentários neutros.')
print('- Muitos comentários neutros foram confundidos com negativos ou positivos.')
print('- Isso sugere sobreposição semântica entre as classes.')

In [ ]:
# ── Métricas por classe ─────────────────────────────────────────────────────
print(classification_report(
    df['sentimento_real'],
    df['sentimento_pred'],
    labels=labels_order,
    target_names=labels_pt,
    digits=3
))

In [ ]:
# ── Onde o texto e a nota divergem ──────────────────────────────────────────
# Casos onde o modelo vê sentimento diferente do que a nota sugere
df['diverge'] = df['sentimento_pred'] != df['sentimento_real']

print(f'Casos divergentes: {df["diverge"].sum():,} ({df["diverge"].mean():.1%})')
print()

# Exemplos mais interessantes: nota alta mas texto negativo
positivo_mas_negativo = df[
    (df['sentimento_real'] == 'positive') &
    (df['sentimento_pred'] == 'negative')
].nlargest(5, 'sentimento_score')

print('── Nota alta (4-5) mas texto negativo ──')
for _, row in positivo_mas_negativo.iterrows():
    print(f'  Nota: {row["review_score"]} | Confiança: {row["sentimento_score"]:.0%}')
    print(f'  Texto: {row["texto"][:120]}')
    print()

# Exemplos: nota baixa mas texto positivo
negativo_mas_positivo = df[
    (df['sentimento_real'] == 'negative') &
    (df['sentimento_pred'] == 'positive')
].nlargest(5, 'sentimento_score')

print('── Nota baixa (1-2) mas texto positivo ──')
for _, row in negativo_mas_positivo.iterrows():
    print(f'  Nota: {row["review_score"]} | Confiança: {row["sentimento_score"]:.0%}')
    print(f'  Texto: {row["texto"][:120]}')
    print()

In [ ]:
# ════════════════════════════════════════════════════════════════
# BLOCO FINAL — Wilcoxon + Tabela Resumo + Gráficos Comparativos
# ════════════════════════════════════════════════════════════════

from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Wilcoxon: TF-IDF vs BERT + TF-IDF ───────────────────────
erros_tfidf = np.abs(resultados[0]['preds'] - y_test.values)
erros_bert  = np.abs(resultados[2]['preds'] - y_test.values)

stat, p = wilcoxon(erros_tfidf, erros_bert)

print("── Teste de Wilcoxon: TF-IDF vs BERT + TF-IDF ─────────────")
print(f"  Estatística : {stat:.2f}")
print(f"  p-value     : {p:.2e}")
print()
if p < 0.05:
    print("  ✅ Diferença estatisticamente significativa (p < 0.05)")
    print("     BERT + TF-IDF é comprovadamente melhor que TF-IDF puro.")
else:
    print("  ❌ Diferença NÃO significativa (p >= 0.05)")

print()

# ── 2. Tabela resumo final ───────────────────────────────────────
print("── Resumo Final — Todos os Modelos ────────────────────────────────────────────")
print(f"  {'Modelo':<35} {'Pearson':>9} {'Spearman':>10} {'RMSE':>8} {'MAE':>8} {'Ac.±1':>8}")
print("  " + "─" * 82)
for r in resultados:
    flag = " ✅" if r['nome'] == 'BERT + TF-IDF' else ""
    print(f"  {r['nome']:<35} {r['pearson']:>9.4f} {r['spearman']:>10.4f} "
          f"{r['rmse']:>8.4f} {r['mae']:>8.4f} {r['acc1']:>8.2%}{flag}")

print()

# ── 3. Gráficos comparativos — Pearson, Spearman, RMSE, Ac.±1 ───
nomes_curtos = [
    'TF-IDF', 'BERTimbau', 'BERT+\nTF-IDF', 'BERT+TF-IDF\n+VADER',
    'BERT+TF-IDF\n+LeIA', 'BERT+TF-IDF\n+TextBlob', 'BERT+TF-IDF\n+Todos',
    'Ridge+TF-IDF\n(Etapa 2)'
]

pearsons  = [r['pearson']  for r in resultados]
spearmans = [r['spearman'] for r in resultados]
rmses     = [r['rmse']     for r in resultados]
maes      = [r['mae']      for r in resultados]
acc1s     = [r['acc1']     for r in resultados]

x          = np.arange(len(resultados))
idx_melhor = 2   # BERT + TF-IDF
cores      = ['#CBD5E1'] * len(resultados)
cores[idx_melhor] = '#06D6A0'
cores[-1]         = '#94A3B8'  # Etapa 2

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Comparação Final de Modelos', fontsize=16, fontweight='bold')

metricas = [
    (axes[0,0], pearsons,          'Pearson r (↑ melhor)',  True),
    (axes[0,1], spearmans,         'Spearman ρ (↑ melhor)', True),
    (axes[1,0], rmses,             'RMSE (↓ melhor)',        False),
    (axes[1,1], [v*100 for v in acc1s], 'Acurácia ±1 % (↑ melhor)', True),
]

for ax, vals, titulo, maior_melhor in metricas:
    bars = ax.bar(x, vals, color=cores, edgecolor='white', linewidth=0.8, width=0.65)

    # Linha baseline TF-IDF
    ax.axhline(vals[0], color='#64748B', linewidth=1.2,
               linestyle='--', alpha=0.7, label='baseline TF-IDF')

    # Anotações
    margem = (max(vals) - min(vals)) * 0.015
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2,
                v + margem,
                f'{v:.3f}' if titulo != 'Acurácia ±1 % (↑ melhor)' else f'{v:.1f}%',
                ha='center', va='bottom', fontsize=7.5, color='#1E293B')

    # Destaque modelo escolhido
    bars[idx_melhor].set_edgecolor('#059669')
    bars[idx_melhor].set_linewidth(2.5)

    ax.set_xticks(x)
    ax.set_xticklabels(nomes_curtos, fontsize=7.5, rotation=20, ha='right')
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_ylim(min(vals) * 0.97, max(vals) * 1.07)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── 4. p-value do Wilcoxon no gráfico ───────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
fig.suptitle('Wilcoxon: Distribuição do Erro Absoluto\nTF-IDF vs BERT + TF-IDF',
             fontsize=12, fontweight='bold')

ax.hist(erros_tfidf, bins=40, alpha=0.6, color='#94A3B8',
        edgecolor='white', label='TF-IDF')
ax.hist(erros_bert,  bins=40, alpha=0.6, color='#06D6A0',
        edgecolor='white', label='BERT + TF-IDF')

ax.axvline(erros_tfidf.mean(), color='#64748B', linewidth=1.5,
           linestyle='--', label=f'média TF-IDF = {erros_tfidf.mean():.3f}')
ax.axvline(erros_bert.mean(),  color='#059669', linewidth=1.5,
           linestyle='--', label=f'média BERT+TF-IDF = {erros_bert.mean():.3f}')

ax.set_xlabel('Erro Absoluto (|predito − real|)')
ax.set_ylabel('Frequência')
ax.legend(fontsize=9)
ax.text(0.98, 0.95, f'p-value = {p:.2e}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#059669', alpha=0.9))

plt.tight_layout()
plt.show()

---
## 4.1 Conclusão da Etapa 4

### O que fizemos
Classificamos cada comentário em **positivo, negativo ou neutro** usando um modelo de linguagem pré-treinado multilíngue (`cardiffnlp/twitter-xlm-roberta-base-sentiment`), sem necessidade de fine-tuning.

### Comparação: sentimento do texto vs nota declarada
A nota do cliente foi usada como referência (1-2 = negativo, 3 = neutro, 4-5 = positivo). Os casos onde o modelo de sentimento discorda da nota revelam situações onde o cliente deu uma nota que não reflete o tom do texto — por exemplo, nota 5 com texto reclamando, ou nota 1 com texto elogiando.

### Por que isso importa
A nota é um clique — rápido e às vezes impreciso. O texto é onde o cliente realmente expressa o que sentiu. Um modelo que lê o texto pode identificar insatisfação mesmo quando a nota não reflete isso, o que é útil para equipes de atendimento e qualidade.

In [ ]:
import os
os.listdir()

In [ ]:
!jupyter nbconvert --to html "Cópia_de_analise_de_sentimento.ipynb"

In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if '.ipynb' in f and ('sentimento' in f.lower() or 'peso' in f.lower() or 'analise' in f.lower()):
            print(repr(os.path.join(root, f)))